## Part 1. Load parking space data

In [2]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv("../data/mtl_parking.csv", encoding="cp1252")
print(df.shape)
df.head()
df.columns.tolist()


(19640, 15)


['sNoPlace',
 'nLongitude',
 'nLatitude',
 'nPositionCentreLongitude',
 'nPositionCentreLatitude',
 'sStatut',
 'sGenre',
 'sType',
 'sAutreTete',
 'sNomRue',
 'nSupVelo',
 'sTypeExploitation',
 'nTarifHoraire',
 'sLocalisation',
 'nTarifMax']

## Part 2. Clean the data
Potential parking space candidates for conversion only include off-street parking (parking lots), so I need to keep every row that is off-street ("H" under the Parking Location column)

Checking how many rows under "S" and "H" parking lcoation. "S" = sur rue (on street parking), "H" = hors rue (off street)

In [3]:
df["sLocalisation"].value_counts().reindex(["S", "H"], fill_value=0)

sLocalisation
S    17962
H     1678
Name: count, dtype: int64

In [4]:
df_h = df[df["sLocalisation"] == "H"] # this keeps only off-street parking spaces

## Join the parking points to their borough
Using a Montreal boundaries GeoJson file

Check which column is associated with the borough names after reading the boundaries dataset:

In [5]:
import pandas as pd
import geopandas as gpd

In [6]:
boroughs = gpd.read_file("../data/boundaries.geojson")
print(boroughs.columns.tolist())

['CODEID', 'NOM', 'NOM_OFFICIEL', 'CODEMAMH', 'CODE_3C', 'NUM', 'ABREV', 'TYPE', 'COMMENT', 'DATEMODIF', 'geometry']


In [7]:
# Create points from the filtered rows (in df_h)
points_h = gpd.GeoDataFrame(
    df_h,
    geometry=gpd.points_from_xy(df_h["nLongitude"], df_h["nLatitude"]),
    crs="EPSG:4326"
)

# Ensure both layers use the same coordinate reference system (CRS)
boroughs = boroughs.to_crs(points_h.crs)

# Assign a borough to each H point
points_h_b = gpd.sjoin(
    points_h,
    boroughs[["NOM", "geometry"]],
    how="left",
    predicate="within"
)

## Part 3. Write out the rows as GeoJSON features and export into parking.geojson

In [8]:
def row_to_feature(row):
    """Convert one parking-table row into a GeoJSON feature."""
    return {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [
                float(row["nLongitude"]),
                float(row["nLatitude"])
            ]
        },
        "properties": {
            "Borough": row["NOM"],   # renaming
            "Hourly rate": row["nTarifHoraire"],
            "Street Name": row["sNomRue"]
        }
    }

In [9]:
features = []
for _, row in points_h_b.iterrows():
    features.append(row_to_feature(row))  # add row_to_feature(row) to the list

collection = {"type": "FeatureCollection", "features": features}

out_path = "../data/parking.geojson"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(collection, f, separators=(",", ":"), allow_nan=False)
    # separators: no spaces, a smaller file.
    # allow_nan=False: stop with an error if any value is NaN (an empty cell).
    # Browsers cannot read NaN in JSON, so without this line the file writes
    # fine and the map fails later with a confusing message. Fail here instead.

import os
print(len(features), "features written to", out_path)
print(len(points_h_b), "features in the original table")
print(round(os.path.getsize(out_path) / 1_000_000, 1), "MB")

1678 features written to ../data/parking.geojson
1678 features in the original table
0.4 MB


## Part 5. Check cell

In [10]:
with open("../data/parking.geojson", encoding="utf-8") as f:
    back = json.load(f)

problems = []
feats = back.get("features", [])
if back.get("type") != "FeatureCollection":
    problems.append("The file is not a FeatureCollection.")
if len(feats) != len(points_h_b):
    problems.append(f"The file has {len(feats)} features but the DataFrame has {len(points_h_b)} rows.")
lons = [ft["geometry"]["coordinates"][0] for ft in feats]
lats = [ft["geometry"]["coordinates"][1] for ft in feats]
if feats and not all(-74.0 < x < -73.4 for x in lons):
    problems.append("Some longitudes are outside the island of Montreal. Are longitude and latitude swapped?")
if feats and not all(45.3 < y < 45.8 for y in lats):
    problems.append("Some latitudes are outside the island of Montreal. Are longitude and latitude swapped?")
if feats and not all({"Street Name", "Borough", "Hourly rate"} <= set(ft["properties"]) for ft in feats):
    problems.append("Some features are missing Street Name, Borough, or Hourly rate in their properties.")

print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

All checks passed


Checking if any points have not been matched to a borough and how many boroughs have points

In [11]:
# Number of points matched to each borough
print(points_h_b["NOM"].value_counts(dropna=False))

# Number of distinct boroughs with points
print("Boroughs with points:", points_h_b["NOM"].nunique())

# Points that did not match a borough
print("Unmatched points:", points_h_b["NOM"].isna().sum())

NOM
Verdun                                 780
Rosemont-La Petite-Patrie              450
Outremont                              169
Ville-Marie                            130
Le Plateau-Mont-Royal                   79
Le Sud-Ouest                            39
Ahuntsic-Cartierville                   21
Côte-des-Neiges-Notre-Dame-de-Grâce     10
Name: count, dtype: int64
Boroughs with points: 8
Unmatched points: 0


## Part 6. Second layer (urban heat island, shapefile)
using Lab 4 recipe 3 and 4

In [12]:
gdf = gpd.read_file("../data/icu_20250821/ICU_20250821.shp")      # or .gpkg, or .geojson
print(gdf.crs, len(gdf), sorted(gdf.geom_type.unique()))
print(gdf.columns.tolist())
gdf = gdf[["TempMoyVF", "ClassesNB_", "ClassQual", "geometry"]]
# Lines or areas only: simplify to about 5 metres if the file is in metres (skip if gdf.crs says degrees)
gdf["geometry"] = gdf.geometry.simplify(5)
gdf = gdf.to_crs("EPSG:4326")
gdf.to_file("../data/heat_island.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

EPSG:2950 556967 ['Polygon']
['gridcode', 'Type_sol', 'TempMoyVF', 'ClassesNB_', 'ClassQual', 'Shape_Leng', 'Shape_Area', 'geometry']


In [13]:
borough_names = [
    "Verdun",
    "Rosemont-La Petite-Patrie",
    "Outremont",
    "Ville-Marie",
    "Le Plateau-Mont-Royal",
    "Le Sud-Ouest",
    "Ahuntsic-Cartierville",
    "Côte-des-Neiges-Notre-Dame-de-Grâce",
]

print("boroughs before:", len(boroughs))
boroughs = boroughs[boroughs["NOM"].isin(borough_names)].copy()
print("boroughs after:", len(boroughs))

# Match coordinate systems, then keep only the heat-layer areas inside those boundaries
boroughs = boroughs.to_crs(gdf.crs)

print("heat polygons before:", len(gdf))
gdf = gpd.clip(gdf, boroughs)
print("heat polygons after:", len(gdf))
# gdf = gdf[gdf["BOROUGH_COLUMN"] == "Plateau-Mont-Royal"]                 # filter by a value
# # or by year:   gdf = gdf[pd.to_datetime(gdf["DATE_COLUMN"]).dt.year == 2025]
# # or by a box (in degrees, after to_crs 4326):   gdf = gdf.cx[-73.62:-73.55, 45.50:45.55]
# print("after:", len(gdf))

boroughs before: 34
boroughs after: 8
heat polygons before: 556967
heat polygons after: 130714


In [14]:
print(gdf[["TempMoyVF", "ClassesNB_", "ClassQual"]].dtypes)
display(gdf[["TempMoyVF", "ClassesNB_", "ClassQual"]].head(10))

print(gdf[["TempMoyVF", "ClassesNB_"]].describe())
print(gdf["ClassQual"].value_counts(dropna=False))

TempMoyVF     float64
ClassesNB_    float64
ClassQual         str
dtype: object


,TempMoyVF,ClassesNB_,ClassQual
441640,26.5238,2.0,Température plus froide que la moyenne
439560,25.5554,2.0,Température plus froide que la moyenne
435375,26.5239,2.0,Température plus froide que la moyenne
435374,26.7224,2.0,Température plus froide que la moyenne
434332,27.1033,2.0,Température plus froide que la moyenne
434331,27.0196,2.0,Température plus froide que la moyenne
433285,26.0529,2.0,Température plus froide que la moyenne
432248,25.8936,2.0,Température plus froide que la moyenne
434330,26.5626,2.0,Température plus froide que la moyenne
434329,26.8414,2.0,Température plus froide que la moyenne


           TempMoyVF     ClassesNB_
count  130714.000000  130714.000000
mean       28.433075       2.916994
std         5.876581       1.113172
min         0.000000       0.000000
25%        26.613775       2.000000
50%        29.063150       3.000000
75%        31.448700       4.000000
max        43.156489       5.000000
ClassQual
Température proche de la moyenne          50181
Zones à risque                            29818
Température plus froide que la moyenne    28143
Îlot de fraicheur                         10761
Îlot de chaleur                            8886
Eau                                        2925
Name: count, dtype: int64


In [15]:
points_for_join = points_h_b.drop(columns=["index_right"], errors="ignore").copy()
heat_for_join = gdf[["ClassQual", "TempMoyVF", "geometry"]].copy()
heat_for_join = heat_for_join.drop(columns=["index_right"], errors="ignore")

if points_for_join.crs != heat_for_join.crs:
    points_for_join = points_for_join.to_crs(heat_for_join.crs)

joined = gpd.sjoin(
    points_for_join,
    heat_for_join,
    how="left",
    predicate="within"
)
joined.to_file("../data/parking_boroughs_heat.geojson", driver="GeoJSON")

In [16]:
import numpy as np
import geopandas as gpd
from shapely.geometry import box

def make_grid(gdf, size):
    """Create square grid cells of `size` metres over the bounds of gdf."""
    minx, miny, maxx, maxy = gdf.total_bounds
    xs = np.arange(np.floor(minx / size) * size, maxx, size)
    ys = np.arange(np.floor(miny / size) * size, maxy, size)
    squares = [box(x, y, x + size, y + size) for x in xs for y in ys]
    return gpd.GeoDataFrame(
        {"cell_id": range(len(squares))},
        geometry=squares,
        crs=gdf.crs,
    )

In [17]:
# Project both layers to metres
boroughs_m = boroughs.to_crs("EPSG:2950")
points_m = points_h_b.to_crs("EPSG:2950")

# Make a 1 km grid over the selected boroughs, then trim cells at their boundaries
grid = make_grid(boroughs_m, 1000)
grid = gpd.clip(grid, boroughs_m)

print(len(grid), "grid cells")

203 grid cells


In [18]:
# Combine clipped pieces that share a cell_id, if the grid was clipped to boroughs
grid_cells = grid.dissolve(by="cell_id", as_index=False)

# Put all layers in the grid's CRS (EPSG:2950, metres)
points_for_grid = (
    points_h_b.to_crs(grid_cells.crs)
    .drop(columns=["index_right"], errors="ignore")
    .reset_index(drop=True)
)
heat_for_grid = gdf[["TempMoyVF", "geometry"]].to_crs(grid_cells.crs)

# Count parking points in each cell
point_joined = gpd.sjoin(
    points_for_grid,
    grid_cells[["cell_id", "geometry"]],
    how="inner",
    predicate="intersects",
)
point_joined = point_joined[~point_joined.index.duplicated()]
point_counts = (
    point_joined.groupby("cell_id")
    .size()
    .rename("parking_points")
    .reset_index()
)

# Intersect the heat polygons with the grid and calculate average temperature per cell
heat_parts = gpd.overlay(
    heat_for_grid,
    grid_cells[["cell_id", "geometry"]],
    how="intersection",
)
heat_parts["area_m2"] = heat_parts.geometry.area
heat_parts = heat_parts[heat_parts["area_m2"] > 0].copy()
heat_parts["temp_area"] = heat_parts["TempMoyVF"] * heat_parts["area_m2"]

heat_summary = (
    heat_parts.groupby("cell_id")
    .agg(
        heat_area_m2=("area_m2", "sum"),
        temp_area=("temp_area", "sum"),
    )
    .reset_index()
)
heat_summary["mean_temp"] = heat_summary["temp_area"] / heat_summary["heat_area_m2"]

# Keep every grid cell; cells without points get a count of zero
cells = (
    grid_cells
    .merge(point_counts, on="cell_id", how="left")
    .merge(heat_summary[["cell_id", "mean_temp"]], on="cell_id", how="left")
)
cells["parking_points"] = cells["parking_points"].fillna(0).astype(int)

print("Grid cells:", len(cells))
print("Parking points counted:", cells["parking_points"].sum(), "of", len(points_h_b))
cells.head()

Grid cells: 203
Parking points counted: 1678 of 1678


,cell_id,geometry,parking_points,mean_temp
0,8,"POLYGON ((285000 5041000, 285000 5040831.041, ...",0,13.318720
1,9,"POLYGON ((285000 5042000, 285000 5041000, 2845...",0,16.200068
2,10,"POLYGON ((285000 5042000, 284747.025 5042000, ...",0,19.041186
3,26,"POLYGON ((285000 5041000, 285331.691 5041000, ...",0,22.329795
4,27,"POLYGON ((285000 5041000, 285000 5042000, 2860...",0,21.553432


In [19]:
import os

# Export the grid with parking-point counts and average heat temperature
web_cells = cells[
    ["cell_id", "parking_points", "mean_temp", "geometry"]
].to_crs("EPSG:4326")
grid_file = "../data/borough_heat_grid.geojson"
web_cells.to_file(grid_file, driver="GeoJSON", COORDINATE_PRECISION=6)

# Export your parking points
parking_web = (
    points_h_b
    .drop(columns=["index_right"], errors="ignore")
    .to_crs("EPSG:4326")
)
points_file = "../data/parking_points.geojson"
parking_web.to_file(points_file, driver="GeoJSON", COORDINATE_PRECISION=6)

for f in [grid_file, points_file]:
    print(f, round(os.path.getsize(f) / 1_000_000, 2), "MB")

../data/borough_heat_grid.geojson 0.2 MB
../data/parking_points.geojson 0.96 MB
